In [1]:
import pandas as pd
downloaded_books = pd.read_csv('downloaded_books.csv')
downloaded_books.head()

,title,title_id
0,Abarat,lit324
1,AbrahamLincolnVampireHunter,lit334
2,AbsalomAbsalom,lit336
3,ACanticleForLeibowitz,lit19
4,ACaribbeanMystery,lit20


In [2]:
lit_tropes = pd.read_csv('../data/tvtropes/lit_tropes.csv')
print(f'all lit tropes: {len(lit_tropes)}')
lit_tropes = lit_tropes[lit_tropes['title_id'].isin(downloaded_books['title_id'])]
print(f'lit tropes in downloaded books: {len(lit_tropes)}')

all lit tropes: 679618
lit tropes in downloaded books: 313883


In [3]:
trope_descriptions = pd.read_csv('../data/tvtropes/tropes.csv')
trope_descriptions.columns, lit_tropes.columns


(Index(['Unnamed: 0', 'TropeID', 'Trope', 'Description'], dtype='object'),
 Index(['Unnamed: 0', 'Title', 'Trope', 'Example', 'trope_id', 'title_id'], dtype='object'))

In [4]:
trope_descriptions = trope_descriptions[trope_descriptions['TropeID'].isin(lit_tropes['trope_id'])]
trope_descriptions.rename(columns={'TropeID': 'trope_id'}, inplace=True)
lit_tropes = lit_tropes.merge(trope_descriptions[['trope_id', 'Description']], on='trope_id', how='left')
lit_tropes.rename(columns={'Description': 'trope_definition', 'Trope': 'trope_name', 'Example': 'trope_example', 'title_id': 'book_id'}, inplace=True)
lit_tropes.drop(columns=['Unnamed: 0'], inplace=True)
lit_tropes.head()

,Title,trope_name,trope_example,trope_id,book_id,trope_definition
0,ACanticleForLeibowitz,Mutants,"After the Flame Deluge, mutations became comm...",t14833,lit19,"\nMutants, also known as Deviants and Freaks, ..."
1,ACanticleForLeibowitz,JustBeforeTheEnd,The third part of the book starts after the i...,t12180,lit19,It's probably sometime between Next Sunday A.D...
2,ACanticleForLeibowitz,JustifiedTitle,Lebowitz is canonized over the course of the ...,t12200,lit19,A strange title naming trope where a title tha...
3,ACanticleForLeibowitz,KillSat,The Asian space platforms that destroy Texark...,t12358,lit19,"When it absolutely, positively, has to be dest..."
4,ACanticleForLeibowitz,PosthumousCharacter,"Saint Leibowitz, though just how posthumous h...",t17441,lit19,"\nA character, dead from the start or killed v..."


In [5]:
print(len(lit_tropes))
# check where trope_example len is under 100 characters
short_examples = lit_tropes[lit_tropes['trope_example'].str.len() < 4]
print(f"Number of trope examples under 100 characters: {len(short_examples)}")
print("\nSample of short trope examples:")
short_examples[['Title', 'trope_name', 'trope_example']].head()

313883
Number of trope examples under 100 characters: 4

Sample of short trope examples:


,Title,trope_name,trope_example
73548,EveryDay,NoBiologicalSex,A.
73572,EveryDay,DidNotGetTheGirl,A.
177036,SeptimusHeap,ArcNumber,7.
302958,WarmBodies,TheBigGuy,M.


In [6]:
# sorting by title_ids with most trope examples
# Count the number of tropes per book
trope_counts = lit_tropes.groupby('Title').size().reset_index(name='trope_count')

# Sort by trope count in descending order
trope_counts = trope_counts.sort_values('trope_count', ascending=True)

# Find books that have a moderate amount of tropes (between 10 and 100)
interesting_books = trope_counts[(trope_counts['trope_count'] > 10) & (trope_counts['trope_count'] < 60)]

specific_tropes = pd.read_json("trope_data/analyzed_tropes.jsonl", lines=True, encoding="utf-8")
specific_tropes = specific_tropes.rename(columns={"trope": "trope_name"})

interesting_tropes = lit_tropes[lit_tropes["Title"].isin(interesting_books["Title"]) & lit_tropes["trope_name"].isin(specific_tropes["trope_name"])]
interesting_tropes = interesting_tropes.rename(columns={"Title": "title"})
interesting_tropes = interesting_tropes.drop_duplicates(subset=["title", "trope_name"])

interesting_tropes = interesting_tropes.merge(specific_tropes, on="trope_name", how="left")

interesting_tropes.to_csv("tropes_dataset.csv")

print(f"Number of books with moderate trope counts (10-100): {len(interesting_books)}")
print(f"Final dataset size: {len(interesting_tropes)}")
interesting_tropes


Number of books with moderate trope counts (10-100): 1032
Final dataset size: 834


,title,trope_name,trope_example,trope_id,book_id,trope_definition,category,specificity,understanding
0,AConspiracyOfPaper,WrongfulAccusationInsurance,"Benjamin Weaver, the protagonist of David Lis...",t26337,lit45,"In detective stories and thrillers, sometimes ...",Storyline,0.8,0.7
1,ADarkerShadeOfMagic,ReturningTheHandkerchief,A Darker Shade of Magic: Lila first gives Kel...,t18749,lit56,\nA common Ship Tease device and an occasional...,Storyline,0.7,0.5
2,ADarkerShadeOfMagic,PossessionBurnout,"In A Darker Shade of Magic, bodies possessed ...",t17422,lit56,"When a demon, ghost, or other body stealing / ...",Storyline,0.8,0.6
3,ADeepnessInTheSky,LetNoCrisisGoToWaste,The Exiled fleet from Vernor Vinge's A Deepne...,t12790,lit62,When one turns an unfortunate and unexpected s...,Character,0.7,0.7
4,ADeepnessInTheSky,FictionAsCoverUp,"In A Deepness in the Sky, humans are hiding i...",t07927,lit62,We've all seen Close Encounters of the Third K...,Storyline,0.9,0.7
...,...,...,...,...,...,...,...,...,...
829,YoungSamurai,MistakenForAliens,Or at least for yokai . Early in Ring of Ea...,t14293,lit15421,\nWhen characters are mistaken for the beings ...,Storyline,0.6,0.4
830,ZForZachariah,SanityBall,Ann is definitely the only one holding this.,t19297,lit15443,When characters on the same show take turns be...,Character,0.7,0.6
831,Zeroes,EmotionControl,"In Zeroes, Bellwether and Mob both have power...",t06736,lit15464,Controlling emotions is quite different from r...,Character,0.9,0.6
832,ZooCity,DoingInTheScientist,Scientists attempt to explain the Undertow an...,t05956,lit15488,This is where a story element (or possibility)...,Narrative,0.8,0.7


In [8]:

import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import random

interesting_tropes['specificity'] = pd.to_numeric(interesting_tropes['specificity'], errors='coerce')
interesting_tropes['understanding'] = pd.to_numeric(interesting_tropes['understanding'], errors='coerce')
interesting_tropes.dropna(subset=['specificity', 'understanding', 'category'], inplace=True)

books_summary = interesting_tropes.groupby('title').agg(
    category_mode=('category', lambda x: x.mode()[0] if not x.mode().empty else 'Unknown'),
    specificity_mean=('specificity', 'mean'),
    understanding_mean=('understanding', 'mean'),
    trope_count=('trope_name', 'size')
).reset_index()

min_books_required = 50
if len(books_summary) < min_books_required:
    print(f"Warning: Only {len(books_summary)} unique books available, cannot select {min_books_required}.")
    n_select = len(books_summary)
else:
    n_select = min_books_required

if n_select == 0:
     raise ValueError("No books available for selection after processing.")

try:
    books_summary['specificity_bin'] = pd.qcut(books_summary['specificity_mean'], q=3, labels=False, duplicates='drop')
    books_summary['understanding_bin'] = pd.qcut(books_summary['understanding_mean'], q=3, labels=False, duplicates='drop')
except ValueError as e:
     print(f"Warning: Could not create bins for stratification due to insufficient unique values: {e}. Proceeding without stratification for binning.")
     books_summary['specificity_bin'] = 0
     books_summary['understanding_bin'] = 0


books_summary['stratify_col'] = (
    books_summary['category_mode'].astype(str) + '_' +
    books_summary['specificity_bin'].astype(str) + '_' +
    books_summary['understanding_bin'].astype(str)
)

unique_strata_counts = books_summary['stratify_col'].value_counts()
if any(unique_strata_counts < 2) and len(books_summary) > n_select:
    print("Warning: Some strata have only one member. Removing them for stratified sampling.")
    single_member_strata = unique_strata_counts[unique_strata_counts < 2].index
    books_summary_filtered = books_summary[~books_summary['stratify_col'].isin(single_member_strata)]
    n_select = min(n_select, len(books_summary_filtered))
    if n_select < min_books_required:
       print(f"Warning: After filtering single-member strata, only {n_select} books can be selected.")
    if n_select == 0:
        raise ValueError("No books left after filtering single-member strata.")
    books_to_sample_from = books_summary_filtered
else:
    books_to_sample_from = books_summary

if len(books_to_sample_from) < n_select:
     print(f"Warning: After processing, only {len(books_to_sample_from)} books are available. Selecting all.")
     n_select = len(books_to_sample_from)

selected_book_titles = []
if n_select > 0:
    try:
        can_stratify = all(books_to_sample_from['stratify_col'].value_counts() >= 2) and len(books_to_sample_from['stratify_col'].unique()) > 1

        if can_stratify:
            selected_books_df, _ = train_test_split(
                books_to_sample_from,
                train_size=n_select,
                test_size=None,
                stratify=books_to_sample_from['stratify_col'],
                random_state=42
            )
            print(f"Selected {n_select} books using stratification.")
        else:
             print(f"Warning: Cannot stratify effectively. Performing random sampling.")
             selected_books_df = books_to_sample_from.sample(n=n_select, random_state=42)
             print(f"Selected {n_select} books randomly.")

        selected_book_titles = selected_books_df['title'].tolist()

    except ValueError as e:
        print(f"Sampling failed: {e}. Performing simple random sampling instead.")
        selected_books_df = books_to_sample_from.sample(n=n_select, random_state=42)
        selected_book_titles = selected_books_df['title'].tolist()
        print(f"Selected {len(selected_book_titles)} books randomly.")
else:
    print("No books were selected.")

train_book_titles = []
test_book_titles = []
if selected_book_titles:
    if len(selected_book_titles) >= 2:
        train_book_titles, test_book_titles = train_test_split(
            selected_book_titles,
            train_size=0.8,
            test_size=0.2,
            random_state=42
        )
        print(f"Split into {len(train_book_titles)} training books and {len(test_book_titles)} testing books.")
    elif len(selected_book_titles) == 1:
        print("Warning: Only one book selected. Assigning it to the training set.")
        train_book_titles = selected_book_titles
        test_book_titles = []
    else:
        print("Warning: No books available for train/test split.")

else:
    print("Cannot create train/test split as no books were selected.")

positive_train_df = interesting_tropes[interesting_tropes['title'].isin(train_book_titles)].copy()
positive_test_df = interesting_tropes[interesting_tropes['title'].isin(test_book_titles)].copy()

positive_train_df['contains'] = True
positive_test_df['contains'] = True

all_trope_details = interesting_tropes.drop_duplicates(subset=['trope_name'])[[
    'trope_name', 'trope_id', 'trope_definition', 'category', 'specificity', 'understanding'
]].set_index('trope_name')

book_id_map = interesting_tropes.drop_duplicates(subset=['title'])[['title', 'book_id']].set_index('title')

all_trope_names = set(all_trope_details.index)




train_df_final = pd.concat([positive_train_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df_final = pd.concat([positive_test_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
train_df_final.to_csv("trope_data/train_df.csv", index=False)
test_df_final.to_csv("trope_data/test_df.csv", index=False)

print("\n--- Final Augmented Datasets ---")

print("\nTraining Set Sample:")
print(train_df_final.head())
print(f"\nTraining Set Shape: {train_df_final.shape}")
print("\nTraining Set 'contains' distribution:")
print(train_df_final['contains'].value_counts(normalize=True))

print("\nTesting Set Sample:")
print(test_df_final.head())
print(f"\nTesting Set Shape: {test_df_final.shape}")
print("\nTesting Set 'contains' distribution:")
print(test_df_final['contains'].value_counts(normalize=True))



print("\n--- Distribution Checks ---")
print("\nTraining Set Category Distribution (Overall):")
print(train_df_final['category'].value_counts(normalize=True))
print("\nTesting Set Category Distribution (Overall):")
print(test_df_final['category'].value_counts(normalize=True))

print("\nTraining Set Specificity Mean:", train_df_final['specificity'].mean())
print("Testing Set Specificity Mean:", test_df_final['specificity'].mean())

print("\nTraining Set Understanding Mean:", train_df_final['understanding'].mean())
print("Testing Set Understanding Mean:", test_df_final['understanding'].mean())



Selected 50 books using stratification.
Split into 40 training books and 10 testing books.

--- Final Augmented Datasets ---

Training Set Sample:
               title            trope_name  \
0  ADeepnessInTheSky  LetNoCrisisGoToWaste   
1       BrightonRock       ReligionIsWrong   
2       TheEgyptGame     ProtectedByAChild   
3          LostSouls        SouthernGothic   
4          Excession       WaxMuseumMorgue   

                                       trope_example trope_id   book_id  \
0   The Exiled fleet from Vernor Vinge's A Deepne...   t12790     lit62   
1   Pinkie interprets Catholicism to mean that  y...   t18571   lit1560   
2   A nonviolent version  Ken and Toby threaten t...   t17837  lit11075   
3   The book takes place in North Carolina and Ne...   t20938   lit6134   
4   In Iain Banks' Excession, the Culture has man...   t25613   lit3421   

                                    trope_definition   category  specificity  \
0  When one turns an unfortunate and unexpect